# Stage 09 — Source-Site Host Cluster Sensitivity

## Scientific role

This is the **single authorized scientific sensitivity extension** after the frozen canonical pipeline:

**Stage 05 FINAL → Stage 06 FINAL → Stage 07 FINAL → Stage 08 FINAL2 → Stage 09 sensitivity**

Stages 05–08 are immutable and are never overwritten.

The purpose is narrow: determine whether source-side uncertainty is materially sensitive to replacing the canonical BanglaBait TEST **row-level** bootstrap cluster with a verified **source-site hostname** cluster derived from the frozen BanglaBait `domain` article-link field.

This notebook does **not** create a new hypothesis, estimand, model, calibration procedure, prediction, cue family, target dataset, bridge analysis, or replacement inference.

## Pre-frozen cluster rule

Rule version: `BANGLABAIT-SOURCE-SITE-HOST-CLUSTER-v1.0.0`

Mapping:

`Stage-05 source prediction row_id`
→ `Stage-03 source_row_id`
→ exact frozen BanglaBait TEST CSV row
→ raw `domain` article-link field
→ normalized source-site hostname
→ `source_site_host:<hostname>`

`domain` is treated as a source-site URL field, **not** as a verified legal publisher ID.

Normalization is outcome-blind and frozen before any sensitivity inference:

1. Unicode NFKC + strip.
2. Parse as URL; if no scheme is present, parse as a scheme-relative URL.
3. Extract hostname only; ignore path/query/fragment/port.
4. Lowercase and IDNA-normalize.
5. Strip trailing dot.
6. Strip **exactly one leading `www.`**.
7. Preserve every other subdomain; do not collapse to eTLD+1.
8. Reject missing/invalid/dotless/localhost/IP hostnames.
9. If a more direct source/publisher/channel identifier column is present in the frozen TEST schema, hard-stop for methodological review instead of silently preferring `domain`.

No row is silently dropped. Any mapping/title/label/identity failure stops the stage before bootstrap.

## Non-rescue rule

Sensitivity results never upgrade a canonical result. Canonical non-estimability remains canonical non-estimability even if this sensitivity becomes estimable.

No Stage-07 H3→H4 bridge is run.


In [1]:
# 1. Environment, immutable identities, output contract
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
from collections import Counter, defaultdict
import csv, hashlib, ipaddress, json, math, os, platform, re, shutil, sys, time, unicodedata
from urllib.parse import urlsplit

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy.stats import chi2
from sklearn.metrics import roc_auc_score
from statsmodels.stats.multitest import multipletests

STAGE_ID = "09"
STAGE_NAME = "09_SOURCE_PUBLISHER_CLUSTER_SENSITIVITY_vFINAL"
PROTOCOL_VERSION = "SWARABYANJAN-CROSSPLATFORM-v1.0.0"
SENSITIVITY_VERSION = "SWARABYANJAN-SOURCE-SITE-CLUSTER-SENSITIVITY-v1.0.0"
CLUSTER_RULE_VERSION = "BANGLABAIT-SOURCE-SITE-HOST-CLUSTER-v1.0.0"

SEEDS = [42, 137, 2718, 31415, 65537]
MODEL_KEYS = ["ebm", "tfidf_lr", "banglabert", "xlmr"]
CUE_FAMILIES = [
    "curiosity_information_withholding",
    "direct_address_imperative",
    "exaggeration_superlative",
    "interrogative_structure",
    "punctuation_intensity",
    "sensational_emotional_loading",
    "specificity_numerals",
    "vague_deictic_reference",
]
PERT_FAMS = [
    "curiosity_information_withholding",
    "direct_address_imperative",
    "punctuation_intensity",
]
PLATFORMS = ["news", "facebook", "youtube", "twitter"]
BUDGETS = [0.05, 0.10, 0.20]
PRIMARY_BUDGET = 0.10
BOOTSTRAP_REPS = 5000
BOOTSTRAP_SEED = 20260906
FDR_Q = 0.05

EXPECTED_FROZEN_ZIP_SHA256 = "e0de5c224f406fe22d8ad30804217465f789c49c70e1d3afbe00afd28deb7ccf"
EXPECTED_BANGLABAIT_TEST_SHA256 = "68176be61ba8dd01f9f78b6aea2b84c198236c89de5abc98db8c49c496ebd996"
EXPECTED_BANGLABAIT_REPO = "Motahar/banglabait"
EXPECTED_BANGLABAIT_REVISION = "ff22fb3b423952959a80dc5d9c6c3345b21cd145"
EXPECTED_SOURCE_TEST_ROWS = 2784

EXPECTED_STAGE05 = {
    "manifest": "7246bc3a49a5f8d8472005cea9f6062d118431cd67d5dd2c4fd8f2ee31f52c27",
    "checksum": "7171a0e581d8fb4bde2a630f4de631d74962954ee3890b8123d78a720823450a",
    "status": "005399d829bb66870810faa3f140757a7ee3d82eac4bae394fbbe79e701db8e6",
    "canonical_version": "SWARABYANJAN-STAGE05-CANONICAL-vFINAL-1.0.1",
}
EXPECTED_STAGE06 = {
    "manifest": "5a2962014445c723379b07fcf02e601644818c36e6c15fb7ee9478ca34711285",
    "checksum": "5f8d0f7fc9cde5e0f159bf4c5f4ca8387441675407b4c43ab87a8063afac3beb",
    "status": "769f33b4771f3ec55700384ae6db7e9e2feb115d23717b27cbd9c17e4983538d",
    "h4": "299fc2e1394c80e1a090ac14d632e0f502b5b779c459e47dcb4e96130b621462",
}
EXPECTED_STAGE07 = {
    "manifest": "fa8651ff00a3306dfd48ed55d21df3ab2eef2fbd365db33657f5c2413822e714",
    "checksum": "86c8aeb1c4c0062303b44d397fece5908c315ff8860fca59afc624bfdde39f8d",
    "status": "566f063f1bef265c6d377565ee6a22ccc1cbbbaa083f7bad7a89423ec6222280",
}
EXPECTED_STAGE08 = {
    "manifest": "b0ac3b392b2e3d1b5136fda5dbdbcf498e1b89c44cc1d920ff597d6c2967523f",
    "checksum": "ccd850d3dbe144651bb52e8277308e6438e3e3d869b2cfc232b39db96fed3af6",
    "status": "d64ee2649a348bad4fd36ee3c4df79fd071b60807b248f0a7a575f5795bf9f33",
}

SEARCH_ROOTS = []
for x in ["/kaggle/input", "/kaggle/working", "/mnt/data", str(Path.cwd())]:
    p = Path(x)
    if p.exists() and p not in SEARCH_ROOTS:
        SEARCH_ROOTS.append(p)

WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUT_ROOT = WORK_ROOT / "09_results" / STAGE_NAME
if OUT_ROOT.exists() and any(OUT_ROOT.iterdir()):
    raise RuntimeError(f"Refusing to overwrite existing Stage-09 directory: {OUT_ROOT}")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

REQUIRED_OUTPUTS = [
    "SOURCE_PUBLISHER_MAPPING.csv",
    "SOURCE_CLUSTER_DIAGNOSTICS.json",
    "H1_H2_SOURCE_CLUSTER_SENSITIVITY.json",
    "H3_SOURCE_CLUSTER_SENSITIVITY.json",
    "H3P_SOURCE_CLUSTER_SENSITIVITY.json",
    "H4_SOURCE_CLUSTER_SENSITIVITY.json",
    "CLAIM_CONCORDANCE_TABLE.csv",
    "SENSITIVITY_SUMMARY.md",
    "REPRODUCIBILITY_MANIFEST.json",
    "STAGE09_SHA256SUMS.txt",
    "STAGE_STATUS.json",
]

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(path, chunk_size=8*1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                return h.hexdigest()
            h.update(b)

def json_builtin(x):
    if isinstance(x, np.bool_): return bool(x)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.floating):
        return None if not np.isfinite(x) else float(x)
    if isinstance(x, np.ndarray): return [json_builtin(v) for v in x.tolist()]
    if isinstance(x, Path): return str(x)
    if isinstance(x, dict): return {str(k): json_builtin(v) for k,v in x.items()}
    if isinstance(x, (list, tuple)): return [json_builtin(v) for v in x]
    return x

def cjb(obj):
    return json.dumps(json_builtin(obj), sort_keys=True, separators=(",", ":"), ensure_ascii=False, allow_nan=False).encode("utf-8")

def write_json(path, obj):
    Path(path).write_text(
        json.dumps(json_builtin(obj), indent=2, sort_keys=True, ensure_ascii=False, allow_nan=False) + "\n",
        encoding="utf-8",
    )

def write_text(path, text):
    Path(path).write_text(str(text), encoding="utf-8")

def set_status(status, message, details=None):
    payload = {
        "stage_id": STAGE_ID, "stage_name": STAGE_NAME, "status": status,
        "message": message, "timestamp_utc": utc_now(), "details": details or {},
    }
    write_json(OUT_ROOT / "STAGE_STATUS.json", payload)
    print(f"STAGE {STAGE_ID} STATUS: {status} — {message}", flush=True)

def hard_stop(message, details=None):
    set_status("HARD_STOP", message, details)
    raise RuntimeError(message)

def parse_sumfile(path):
    out = {}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if not line.strip(): continue
        parts = line.split(None, 1)
        if len(parts) != 2 or not re.fullmatch(r"[0-9a-fA-F]{64}", parts[0]):
            raise RuntimeError(f"Malformed checksum line: {line!r}")
        rel = parts[1].lstrip("*").strip().replace("\\", "/")
        if rel in out: raise RuntimeError(f"Duplicate checksum path: {rel}")
        out[rel] = parts[0].lower()
    if not out: raise RuntimeError(f"Empty checksum contract: {path}")
    return out

def verify_sumfile(root, sum_name, full=True):
    sums = parse_sumfile(root / sum_name)
    if full:
        for rel, expected in sums.items():
            p = root / rel
            if not p.is_file():
                hard_stop("Checksum-contract artifact missing", {"root": str(root), "artifact": rel})
            if sha256_file(p) != expected:
                hard_stop("Checksum-contract artifact mismatch", {"root": str(root), "artifact": rel})
    return sums

def verify_payload_seal(manifest, label):
    if "manifest_payload_sha256" not in manifest:
        hard_stop(f"{label} manifest payload seal missing")
    recorded = manifest["manifest_payload_sha256"]
    tmp = dict(manifest); tmp.pop("manifest_payload_sha256", None)
    observed = hashlib.sha256(cjb(tmp)).hexdigest()
    if observed != recorded:
        hard_stop(f"{label} manifest payload seal mismatch", {"recorded": recorded, "observed": observed})
    return observed

def discover_exact_root(manifest_name, expected_hash, required_files):
    hits = []
    seen = set()
    for root in SEARCH_ROOTS:
        try:
            candidates = list(root.rglob(manifest_name))
        except Exception:
            candidates = []
        for p in candidates:
            if not p.is_file(): continue
            d = p.parent.resolve()
            if d in seen: continue
            seen.add(d)
            if sha256_file(p) != expected_hash: continue
            if all((d / f).is_file() for f in required_files):
                hits.append(d)
    if not hits:
        hard_stop("Exact canonical upstream root not found", {
            "manifest": manifest_name, "expected_sha256": expected_hash,
            "required_files": required_files, "searched_roots": [str(x) for x in SEARCH_ROOTS],
        })
    return sorted(set(hits), key=str)[0], [str(x) for x in sorted(set(hits), key=str)]

def discover_file_by_hash(name, expected_hash, required_neighbors=()):
    hits = []
    for root in SEARCH_ROOTS:
        try:
            candidates = list(root.rglob(name))
        except Exception:
            candidates = []
        for p in candidates:
            if p.is_file() and sha256_file(p) == expected_hash:
                if all((p.parent / n).exists() for n in required_neighbors):
                    hits.append(p.resolve())
    if not hits:
        hard_stop("Exact hashed upstream file not found", {"name": name, "expected_sha256": expected_hash})
    return sorted(set(hits), key=str)[0]

print("Stage-09 environment/configuration initialized; no scientific computation has run.", flush=True)


Stage-09 environment/configuration initialized; no scientific computation has run.


In [2]:
# 2. Exact canonical upstream provenance gate: Stage03 + Stage05/06/07/08
S5_REQUIRED = [
    "STAGE05_ARTIFACT_MANIFEST.json", "STAGE05_SHA256SUMS.txt", "STAGE_STATUS.json",
    "LOCKED_SOURCE_TEST_PREDICTIONS.parquet", "BANGLACLICK_PLATFORM_PREDICTIONS.parquet",
    "BAITBUSTER_HUMAN_PREDICTIONS.parquet", "H3_PERTURBATION_PAIRS.parquet",
    "H3_PERTURBATION_EXCLUSIONS.csv", "H1_RESULTS.json", "H2_RESULTS.json",
    "H3_RESULTS.json", "H3_PERTURBATION_RESULTS.json", "STATISTICAL_SUMMARY.json",
]
S5, S5_MOUNTS = discover_exact_root("STAGE05_ARTIFACT_MANIFEST.json", EXPECTED_STAGE05["manifest"], S5_REQUIRED)
if sha256_file(S5/"STAGE05_SHA256SUMS.txt") != EXPECTED_STAGE05["checksum"]:
    hard_stop("Stage-05 checksum-file identity mismatch")
if sha256_file(S5/"STAGE_STATUS.json") != EXPECTED_STAGE05["status"]:
    hard_stop("Stage-05 status-file identity mismatch")
S5_MAN = json.loads((S5/"STAGE05_ARTIFACT_MANIFEST.json").read_text(encoding="utf-8"))
verify_payload_seal(S5_MAN, "Stage-05")
if S5_MAN.get("stage_id") != "05" or S5_MAN.get("canonical_stage05") is not True:
    hard_stop("Stage-05 canonical identity mismatch")
if S5_MAN.get("canonical_version") != EXPECTED_STAGE05["canonical_version"]:
    hard_stop("Stage-05 canonical version mismatch")
if S5_MAN.get("protocol_version") != PROTOCOL_VERSION:
    hard_stop("Stage-05 protocol mismatch")
if json.loads((S5/"STAGE_STATUS.json").read_text(encoding="utf-8")).get("status") != "PASS":
    hard_stop("Stage-05 is not PASS")
S5_SUMS = verify_sumfile(S5, "STAGE05_SHA256SUMS.txt", full=True)

S6_REQUIRED = ["REPRODUCIBILITY_MANIFEST.json","STAGE06_SHA256SUMS.txt","STAGE_STATUS.json","H4_RESULTS.json"]
S6, S6_MOUNTS = discover_exact_root("REPRODUCIBILITY_MANIFEST.json", EXPECTED_STAGE06["manifest"], S6_REQUIRED)
if sha256_file(S6/"STAGE06_SHA256SUMS.txt") != EXPECTED_STAGE06["checksum"]: hard_stop("Stage-06 checksum identity mismatch")
if sha256_file(S6/"STAGE_STATUS.json") != EXPECTED_STAGE06["status"]: hard_stop("Stage-06 status identity mismatch")
if sha256_file(S6/"H4_RESULTS.json") != EXPECTED_STAGE06["h4"]: hard_stop("Stage-06 H4 identity mismatch")
S6_MAN = json.loads((S6/"REPRODUCIBILITY_MANIFEST.json").read_text(encoding="utf-8")); verify_payload_seal(S6_MAN,"Stage-06")
if json.loads((S6/"STAGE_STATUS.json").read_text(encoding="utf-8")).get("status") != "PASS": hard_stop("Stage-06 is not PASS")
verify_sumfile(S6, "STAGE06_SHA256SUMS.txt", full=True)
if S6_MAN.get("upstream",{}).get("stage05_manifest_sha256") != EXPECTED_STAGE05["manifest"]:
    hard_stop("Stage-06 does not bind exact canonical Stage-05")

S7_REQUIRED = ["REPRODUCIBILITY_MANIFEST.json","STAGE07_SHA256SUMS.txt","STAGE_STATUS.json"]
S7, S7_MOUNTS = discover_exact_root("REPRODUCIBILITY_MANIFEST.json", EXPECTED_STAGE07["manifest"], S7_REQUIRED)
if sha256_file(S7/"STAGE07_SHA256SUMS.txt") != EXPECTED_STAGE07["checksum"]: hard_stop("Stage-07 checksum identity mismatch")
if sha256_file(S7/"STAGE_STATUS.json") != EXPECTED_STAGE07["status"]: hard_stop("Stage-07 status identity mismatch")
S7_MAN = json.loads((S7/"REPRODUCIBILITY_MANIFEST.json").read_text(encoding="utf-8")); verify_payload_seal(S7_MAN,"Stage-07")
if json.loads((S7/"STAGE_STATUS.json").read_text(encoding="utf-8")).get("status") != "PASS": hard_stop("Stage-07 is not PASS")
verify_sumfile(S7, "STAGE07_SHA256SUMS.txt", full=True)
if S7_MAN.get("upstream",{}).get("stage05_manifest_sha256") != EXPECTED_STAGE05["manifest"]:
    hard_stop("Stage-07 Stage-05 identity mismatch")
if S7_MAN.get("upstream",{}).get("stage06_manifest_sha256") != EXPECTED_STAGE06["manifest"]:
    hard_stop("Stage-07 Stage-06 identity mismatch")

S8_REQUIRED = ["MANIFEST.json","STAGE08_SHA256SUMS.txt","STAGE08_STATUS.json","PROVENANCE_REPORT.json"]
S8, S8_MOUNTS = discover_exact_root("MANIFEST.json", EXPECTED_STAGE08["manifest"], S8_REQUIRED)
if sha256_file(S8/"STAGE08_SHA256SUMS.txt") != EXPECTED_STAGE08["checksum"]: hard_stop("Stage-08 checksum identity mismatch")
if sha256_file(S8/"STAGE08_STATUS.json") != EXPECTED_STAGE08["status"]: hard_stop("Stage-08 status identity mismatch")
S8_MAN = json.loads((S8/"MANIFEST.json").read_text(encoding="utf-8")); verify_payload_seal(S8_MAN,"Stage-08")
if json.loads((S8/"STAGE08_STATUS.json").read_text(encoding="utf-8")).get("status") != "PASS": hard_stop("Stage-08 FINAL2 is not PASS")
verify_sumfile(S8, "STAGE08_SHA256SUMS.txt", full=True)

# Discover Stage-03 through the exact Stage-03 checksum-file identity carried by canonical Stage-05.
S3_SUM_EXPECTED = S5_MAN.get("upstream_stage03_checksum_manifest_sha256")
if not isinstance(S3_SUM_EXPECTED, str) or not re.fullmatch(r"[0-9a-f]{64}", S3_SUM_EXPECTED):
    hard_stop("Canonical Stage-05 does not expose a valid Stage-03 checksum identity")
S3_SUM_PATH = discover_file_by_hash("STAGE03_SHA256SUMS.txt", S3_SUM_EXPECTED)
S3 = S3_SUM_PATH.parent.resolve()
for f in ["MODEL_ARTIFACT_MANIFEST.json","CLEAN_DATA_MANIFEST.json","STAGE_STATUS.json"]:
    if not (S3/f).is_file(): hard_stop("Stage-03 mapping artifact missing", {"artifact":f})
if json.loads((S3/"STAGE_STATUS.json").read_text(encoding="utf-8")).get("status") != "PASS":
    hard_stop("Stage-03 is not PASS")
S3_SUMS = parse_sumfile(S3/"STAGE03_SHA256SUMS.txt")
for f in ["MODEL_ARTIFACT_MANIFEST.json","CLEAN_DATA_MANIFEST.json"]:
    if f not in S3_SUMS or sha256_file(S3/f) != S3_SUMS[f]:
        hard_stop("Stage-03 mapping artifact checksum mismatch", {"artifact":f})
S3_MAN = json.loads((S3/"MODEL_ARTIFACT_MANIFEST.json").read_text(encoding="utf-8")); verify_payload_seal(S3_MAN,"Stage-03 model artifact")
if S3_MAN.get("protocol_version") != PROTOCOL_VERSION or S3_MAN.get("frozen_package_sha256") != EXPECTED_FROZEN_ZIP_SHA256:
    hard_stop("Stage-03 frozen/protocol identity mismatch")
test_rec = S3_MAN.get("clean_split_artifacts",{}).get("test")
if not isinstance(test_rec,dict) or "path" not in test_rec or "sha256" not in test_rec:
    hard_stop("Stage-03 clean TEST artifact identity missing")
S3_CLEAN_TEST = S3 / test_rec["path"]
if not S3_CLEAN_TEST.is_file() or sha256_file(S3_CLEAN_TEST) != test_rec["sha256"]:
    hard_stop("Stage-03 clean TEST artifact mismatch")

UPSTREAM_IDENTITIES = {
    "stage03_checksum_sha256": S3_SUM_EXPECTED,
    "stage05_manifest_sha256": EXPECTED_STAGE05["manifest"],
    "stage05_checksum_sha256": EXPECTED_STAGE05["checksum"],
    "stage05_status_sha256": EXPECTED_STAGE05["status"],
    "stage06_manifest_sha256": EXPECTED_STAGE06["manifest"],
    "stage06_checksum_sha256": EXPECTED_STAGE06["checksum"],
    "stage06_status_sha256": EXPECTED_STAGE06["status"],
    "stage07_manifest_sha256": EXPECTED_STAGE07["manifest"],
    "stage07_checksum_sha256": EXPECTED_STAGE07["checksum"],
    "stage07_status_sha256": EXPECTED_STAGE07["status"],
    "stage08_manifest_sha256": EXPECTED_STAGE08["manifest"],
    "stage08_checksum_sha256": EXPECTED_STAGE08["checksum"],
    "stage08_status_sha256": EXPECTED_STAGE08["status"],
}
print("Exact Stage03 + Stage05→06→07→08 provenance gate: PASS", flush=True)


Exact Stage03 + Stage05→06→07→08 provenance gate: PASS


In [3]:
# 3. Frozen BanglaBait TEST metadata validation, source-site mapping, and diagnostics
DIRECT_ENTITY_ID_COLUMNS = {
    "publisher","publisher_id","publisher_name","source","source_id","source_name",
    "site","site_id","site_name","outlet","outlet_id","outlet_name",
    "channel","channel_id","news_source","news_source_id",
}
MIN_CLUSTERS_TO_RUN = 10
FEW_CLUSTER_WARNING_THRESHOLD = 20
DOMINANT_CLUSTER_SHARE_THRESHOLD = 0.20

def normalize_source_site_host(raw):
    if raw is None:
        raise ValueError("missing_domain")
    text = unicodedata.normalize("NFKC", str(raw)).strip()
    if not text:
        raise ValueError("empty_domain")
    candidate = text if "://" in text else "//" + text
    parts = urlsplit(candidate)
    host = parts.hostname
    if host is None:
        raise ValueError("hostname_unparseable")
    host = unicodedata.normalize("NFKC", host).strip().lower().rstrip(".")
    if host.startswith("www."):
        host = host[4:]
    try:
        host = host.encode("idna").decode("ascii")
    except Exception as exc:
        raise ValueError("hostname_idna_invalid") from exc
    if not host or host == "localhost" or "." not in host:
        raise ValueError("hostname_not_public_site")
    try:
        ipaddress.ip_address(host)
    except ValueError:
        pass
    else:
        raise ValueError("hostname_is_ip")
    if any((not part) or len(part) > 63 or not re.fullmatch(r"[a-z0-9-]+", part) or part.startswith("-") or part.endswith("-")
           for part in host.split(".")):
        raise ValueError("hostname_invalid_dns_syntax")
    return host

# Locate exact frozen BanglaBait TEST bytes; identical duplicate mounts are acceptable.
test_hits = []
for root in SEARCH_ROOTS:
    try:
        candidates = list(root.rglob("test.csv"))
    except Exception:
        candidates = []
    for p in candidates:
        if "BanglaBait" not in p.parts: continue
        try:
            if sha256_file(p) == EXPECTED_BANGLABAIT_TEST_SHA256:
                test_hits.append(p.resolve())
        except Exception:
            pass
if not test_hits:
    hard_stop("Exact frozen BanglaBait TEST bytes not found", {"expected_sha256":EXPECTED_BANGLABAIT_TEST_SHA256})
RAW_TEST = sorted(set(test_hits),key=str)[0]

# Find the frozen package root above the exact TEST file.
FROZEN_ROOT = None
for ancestor in RAW_TEST.parents:
    if all((ancestor/x).is_file() for x in ["SOURCES_LOCK.json","ACQUISITION_STATUS.json","SHA256SUMS.txt"]):
        FROZEN_ROOT = ancestor
        break
if FROZEN_ROOT is None:
    hard_stop("Frozen package controls not found above BanglaBait TEST")

LOCK = json.loads((FROZEN_ROOT/"SOURCES_LOCK.json").read_text(encoding="utf-8"))
bb_lock = LOCK.get("sources",{}).get("BanglaBait",{})
if bb_lock.get("repo_id") != EXPECTED_BANGLABAIT_REPO or bb_lock.get("revision") != EXPECTED_BANGLABAIT_REVISION:
    hard_stop("BanglaBait source revision mismatch")
lock_test = bb_lock.get("expected_tree",{}).get("test.csv",{})
if lock_test.get("sha256") != EXPECTED_BANGLABAIT_TEST_SHA256:
    hard_stop("BanglaBait TEST hash not pinned by SOURCES_LOCK")
ACQ = json.loads((FROZEN_ROOT/"ACQUISITION_STATUS.json").read_text(encoding="utf-8"))
if ACQ.get("datasets",{}).get("BanglaBait",{}).get("status") != "PASS" or ACQ.get("package",{}).get("status") != "PASS":
    hard_stop("Frozen BanglaBait/package acquisition status is not PASS")
frozen_sums = parse_sumfile(FROZEN_ROOT/"SHA256SUMS.txt")
matches = [h for rel,h in frozen_sums.items() if rel.replace("\\","/").endswith("BanglaBait/repository_snapshot/test.csv")]
if matches != [EXPECTED_BANGLABAIT_TEST_SHA256]:
    hard_stop("Frozen SHA256SUMS does not uniquely bind BanglaBait TEST")

# Read raw CSV exactly with the row-number convention used by Stage03.
raw_records = []
with open(RAW_TEST, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    raw_headers = [str(x) for x in (reader.fieldnames or [])]
    header_lower = {unicodedata.normalize("NFKC", x).strip().lower(): x for x in raw_headers}
    for required in ["domain","title","label"]:
        if required not in header_lower:
            hard_stop("Frozen BanglaBait TEST lacks required clustering/mapping column", {"required":required,"headers":raw_headers})
    direct_present = sorted(k for k in DIRECT_ENTITY_ID_COLUMNS if k in header_lower)
    if direct_present:
        hard_stop(
            "Frozen BanglaBait TEST contains a more direct source/publisher/channel identifier; "
            "the pre-frozen domain-host rule must not silently override it.",
            {"direct_identifier_columns":direct_present,"headers":raw_headers},
        )
    domain_col, title_col, label_col = header_lower["domain"], header_lower["title"], header_lower["label"]
    for row_id,row in enumerate(reader):
        raw_records.append({
            "source_row_id": int(row_id),
            "domain": row.get(domain_col),
            "title": row.get(title_col),
            "label_raw": row.get(label_col),
        })
RAW_BY_ID = {r["source_row_id"]:r for r in raw_records}

# Stage03 clean TEST row universe.
clean = pd.read_parquet(S3_CLEAN_TEST).sort_values("source_row_id",kind="stable").reset_index(drop=True)
needed_clean = {"source_row_id","title","label"}
if not needed_clean.issubset(clean.columns):
    hard_stop("Stage-03 clean TEST schema lacks exact mapping fields", {"columns":sorted(clean.columns)})
if len(clean) != EXPECTED_SOURCE_TEST_ROWS or clean["source_row_id"].duplicated().any():
    hard_stop("Stage-03 clean TEST row contract mismatch", {"rows":len(clean),"duplicates":int(clean["source_row_id"].duplicated().sum())})

# Stage05 canonical source prediction identity, collapsed to one static row per source item.
SRC_P_RAW = pd.read_parquet(S5/"LOCKED_SOURCE_TEST_PREDICTIONS.parquet")
source_static = (
    SRC_P_RAW[(SRC_P_RAW["model"]=="ebm") & (SRC_P_RAW["seed"].astype(int)==SEEDS[0])]
    .sort_values("row_id",kind="stable").drop_duplicates("row_id").reset_index(drop=True)
)
if len(source_static) != EXPECTED_SOURCE_TEST_ROWS or source_static["row_id"].duplicated().any():
    hard_stop("Stage-05 source item universe mismatch")
if not source_static["row_id"].astype(str).str.fullmatch(r"\d+").all():
    hard_stop("Stage-05 source row_id is not the frozen Stage03 numeric source_row_id")
source_static["source_row_id"] = source_static["row_id"].astype(int)

clean_ids = set(clean["source_row_id"].astype(int))
pred_ids = set(source_static["source_row_id"].astype(int))
if clean_ids != pred_ids:
    hard_stop("Stage03 clean TEST and Stage05 source prediction row IDs differ", {
        "clean_only":sorted(clean_ids-pred_ids)[:20],"pred_only":sorted(pred_ids-clean_ids)[:20],
    })

mapping_rows = []
mapping_errors = []
clean_lookup = clean.set_index("source_row_id")
pred_lookup = source_static.set_index("source_row_id")
for rid in sorted(clean_ids):
    if rid not in RAW_BY_ID:
        mapping_errors.append({"source_row_id":rid,"reason":"raw_row_missing"})
        continue
    rr = RAW_BY_ID[rid]
    cr = clean_lookup.loc[rid]
    pr = pred_lookup.loc[rid]
    raw_title = "" if rr["title"] is None else str(rr["title"])
    if raw_title != str(cr["title"]):
        mapping_errors.append({"source_row_id":rid,"reason":"title_mismatch_raw_vs_stage03"})
        continue
    if raw_title != str(pr["title"]):
        mapping_errors.append({"source_row_id":rid,"reason":"title_mismatch_raw_vs_stage05"})
        continue
    raw_label = unicodedata.normalize("NFKC", "" if rr["label_raw"] is None else str(rr["label_raw"])).strip()
    if raw_label not in {"0","1"}:
        mapping_errors.append({"source_row_id":rid,"reason":"raw_label_invalid","value":raw_label})
        continue
    if int(raw_label) != int(cr["label"]) or int(raw_label) != int(pr["label"]):
        mapping_errors.append({"source_row_id":rid,"reason":"label_mismatch"})
        continue
    try:
        host = normalize_source_site_host(rr["domain"])
    except ValueError as exc:
        mapping_errors.append({"source_row_id":rid,"reason":str(exc),"domain":rr["domain"]})
        continue
    mapping_rows.append({
        "source_row_id":rid,
        "stage05_row_id":str(pr["row_id"]),
        "raw_domain":str(rr["domain"]),
        "normalized_source_site_host":host,
        "source_cluster_id":"source_site_host:"+host,
        "label":int(pr["label"]),
        "title_sha256":hashlib.sha256(raw_title.encode("utf-8")).hexdigest(),
        "mapping_status":"PASS",
    })

# No row may be silently dropped.
if mapping_errors or len(mapping_rows) != EXPECTED_SOURCE_TEST_ROWS:
    pd.DataFrame(mapping_rows).to_csv(OUT_ROOT/"SOURCE_PUBLISHER_MAPPING.csv",index=False)
    prelim = {
        "schema_version":1,"cluster_rule_version":CLUSTER_RULE_VERSION,
        "mapping_gate":"FAIL","total_clean_source_rows":EXPECTED_SOURCE_TEST_ROWS,
        "mapped_rows":len(mapping_rows),"invalid_or_ambiguous_rows":len(mapping_errors),
        "mapping_errors":mapping_errors[:100],"raw_headers":raw_headers,
    }
    write_json(OUT_ROOT/"SOURCE_CLUSTER_DIAGNOSTICS.json",prelim)
    hard_stop("Source-site mapping failed closed; bootstrap prohibited", {"invalid_rows":len(mapping_errors)})

MAP = pd.DataFrame(mapping_rows).sort_values("source_row_id",kind="stable").reset_index(drop=True)
MAP.to_csv(OUT_ROOT/"SOURCE_PUBLISHER_MAPPING.csv",index=False)

sizes = MAP.groupby("normalized_source_site_host",sort=True).size().sort_values(ascending=False)
n_clusters = int(len(sizes))
if n_clusters < MIN_CLUSTERS_TO_RUN:
    gate = "FAIL_INSUFFICIENT_SOURCE_SITE_CLUSTERS"
else:
    gate = "PASS"
top10 = sizes.head(10)
major_rows = []
for host,n in top10.items():
    g = MAP[MAP["normalized_source_site_host"].eq(host)]
    major_rows.append({
        "normalized_source_site_host":host,"rows":int(n),
        "share":float(n/len(MAP)),"positive_prevalence":float(g["label"].mean()),
        "positive_rows":int(g["label"].sum()),"negative_rows":int((1-g["label"]).sum()),
    })
hist = Counter(map(int,sizes.tolist()))
diag = {
    "schema_version":2,
    "cluster_rule_version":CLUSTER_RULE_VERSION,
    "cluster_semantics":"normalized source-site hostname from BanglaBait domain/article-link field; not asserted to be a legal publisher identity",
    "normalization":{
        "unicode":"NFKC","hostname_only":True,"lowercase":True,"idna":True,
        "strip_trailing_dot":True,"strip_exact_leading_www":True,
        "preserve_other_subdomains":True,"etld_plus_one_collapse":False,
        "ports_paths_queries_fragments_ignored":True,
    },
    "mapping_gate":gate,
    "total_source_test_rows":int(len(MAP)),
    "total_publisher_clusters":n_clusters,
    "cluster_size_distribution":{"histogram_size_to_cluster_count":dict(sorted(hist.items()))},
    "cluster_size_min":int(sizes.min()),
    "cluster_size_median":float(sizes.median()),
    "cluster_size_mean":float(sizes.mean()),
    "cluster_size_max":int(sizes.max()),
    "singleton_clusters":int((sizes==1).sum()),
    "top5_cluster_share":float(sizes.head(5).sum()/len(MAP)),
    "top10_cluster_share":float(sizes.head(10).sum()/len(MAP)),
    "proportion_observations_in_repeated_clusters":float(MAP["normalized_source_site_host"].map(sizes).gt(1).mean()),
    "major_cluster_label_prevalence":major_rows,
    "dominant_cluster_threshold":DOMINANT_CLUSTER_SHARE_THRESHOLD,
    "dominant_cluster_present":bool(float(sizes.iloc[0]/len(MAP)) >= DOMINANT_CLUSTER_SHARE_THRESHOLD),
    "largest_cluster_share":float(sizes.iloc[0]/len(MAP)),
    "few_cluster_warning_threshold":FEW_CLUSTER_WARNING_THRESHOLD,
    "few_cluster_warning":bool(n_clusters < FEW_CLUSTER_WARNING_THRESHOLD),
    "minimum_clusters_to_run":MIN_CLUSTERS_TO_RUN,
    "rows_excluded_due_missing_or_ambiguous_mapping":0,
    "raw_headers":raw_headers,
    "direct_source_identifier_columns_detected":[],
    "representative_mapping_rows":MAP.head(10).to_dict("records"),
    "frozen_test_sha256":EXPECTED_BANGLABAIT_TEST_SHA256,
    "frozen_repo_id":EXPECTED_BANGLABAIT_REPO,
    "frozen_revision":EXPECTED_BANGLABAIT_REVISION,
}
write_json(OUT_ROOT/"SOURCE_CLUSTER_DIAGNOSTICS.json",diag)
if gate != "PASS":
    hard_stop("Source-site cluster count is too small for the pre-frozen ordinary cluster-bootstrap sensitivity", diag)
print(f"Source-site cluster validation: PASS rows={len(MAP)} clusters={n_clusters}",flush=True)


Source-site cluster validation: PASS rows=2784 clusters=40


In [4]:
# 4. Load immutable scientific inputs; replace ONLY source cluster_id in derived in-memory views
SRC_P = pd.read_parquet(S5/"LOCKED_SOURCE_TEST_PREDICTIONS.parquet")
BC_P = pd.read_parquet(S5/"BANGLACLICK_PLATFORM_PREDICTIONS.parquet")
BB_P = pd.read_parquet(S5/"BAITBUSTER_HUMAN_PREDICTIONS.parquet")
PERT = pd.read_parquet(S5/"H3_PERTURBATION_PAIRS.parquet")

CAN_H1 = json.loads((S5/"H1_RESULTS.json").read_text(encoding="utf-8"))
CAN_H2 = json.loads((S5/"H2_RESULTS.json").read_text(encoding="utf-8"))
CAN_H3 = json.loads((S5/"H3_RESULTS.json").read_text(encoding="utf-8"))
CAN_H3P = json.loads((S5/"H3_PERTURBATION_RESULTS.json").read_text(encoding="utf-8"))
CAN_STAT = json.loads((S5/"STATISTICAL_SUMMARY.json").read_text(encoding="utf-8"))
CAN_H4 = json.loads((S6/"H4_RESULTS.json").read_text(encoding="utf-8"))

required_pred = {
    "row_id","title","platform","label","cluster_id","dataset","model","seed",
    "logit","p_uncalibrated","temperature","p_calibrated","normalized_title_sha256",
    "token_count","latin_code_mixing_proportion",*CUE_FAMILIES,
}
for name,df,expected_n in [("source",SRC_P,2784),("banglaclick",BC_P,4961),("baitbuster",BB_P,10000)]:
    if not required_pred.issubset(df.columns):
        hard_stop("Canonical prediction schema incomplete",{"domain":name,"missing":sorted(required_pred-set(df.columns))})
    if set(df["model"].astype(str).unique()) != set(MODEL_KEYS) or set(df["seed"].astype(int).unique()) != set(SEEDS):
        hard_stop("Canonical model/seed panel changed",{"domain":name})
    for (m,s),g in df.groupby(["model","seed"],sort=True):
        if len(g)!=expected_n or g["row_id"].nunique()!=expected_n:
            hard_stop("Canonical prediction row panel changed",{"domain":name,"model":str(m),"seed":int(s)})

cluster_map = dict(zip(MAP["stage05_row_id"].astype(str),MAP["source_cluster_id"].astype(str)))
SRC_SENS = SRC_P.copy()
if set(SRC_SENS["row_id"].astype(str).unique()) != set(cluster_map):
    hard_stop("Source cluster map does not cover canonical source prediction row IDs")
SRC_SENS["canonical_cluster_id"] = SRC_SENS["cluster_id"].astype(str)
SRC_SENS["cluster_id"] = SRC_SENS["row_id"].astype(str).map(cluster_map)
if SRC_SENS["cluster_id"].isna().any():
    hard_stop("Derived source cluster_id contains missing values")
if not SRC_SENS["canonical_cluster_id"].str.startswith("source_row:").all():
    hard_stop("Canonical source cluster rule is not the frozen row-level fallback")
if not SRC_SENS["cluster_id"].str.startswith("source_site_host:").all():
    hard_stop("Sensitivity source cluster rule was not applied")

# Target clusters must remain byte-for-byte equal to canonical Stage05 fields.
if not BC_P["cluster_id"].astype(str).str.startswith("bc:").all(): hard_stop("BanglaClick target cluster identity changed")
if not BB_P["cluster_id"].astype(str).str.startswith("bb:").all(): hard_stop("BaitBuster target cluster identity changed")

PERT_SENS = PERT.copy()
source_mask = PERT_SENS["domain"].eq("source_test")
if not set(PERT_SENS.loc[source_mask,"row_id"].astype(str)).issubset(set(cluster_map)):
    hard_stop("H3-P source pair row IDs are not covered by source-site mapping")
PERT_SENS.loc[source_mask,"cluster_id"] = PERT_SENS.loc[source_mask,"row_id"].astype(str).map(cluster_map)
if PERT_SENS.loc[source_mask,"cluster_id"].isna().any(): hard_stop("H3-P source cluster remap failed")

# Canonical target cluster IDs in H3-P remain unchanged.
if not PERT_SENS.loc[~source_mask,"cluster_id"].astype(str).equals(PERT.loc[~source_mask,"cluster_id"].astype(str)):
    hard_stop("H3-P target cluster identity changed")

# Point-estimate parity helpers and canonical lookups.
CAN_H1_ASSESS = {(r["domain"],r["model"],int(r["seed"])):r for r in CAN_H1["confirmatory_assessment_rows"]}
CAN_H2_ASSESS = {(r["domain"],r["model"],int(r["seed"])):r for r in CAN_H2["confirmatory_assessment_rows"]}
CAN_STAT_ROWS = {(r["domain"],r["model"],int(r["seed"])):r for r in CAN_STAT["h1_h2_bootstrap"]}
CAN_H3_CONTRASTS = {(r["cue"],r["target"]):r for r in CAN_H3["banglaclick_cue_platform_contrasts"]}
CAN_H3P_ROWS = {(r["cue"],r["model"],int(r["seed"]),r["target"]):r for r in CAN_H3P["rows"]}
CAN_H4_ROWS = {(r["domain"],r["model"],int(r["seed"]),float(r["budget"])):r for r in CAN_H4["bootstrap_rows"]}
CAN_H4_POINTS = {(r["domain"],r["model"],int(r["seed"]),float(r["budget"])):r for r in CAN_H4["fixed_budget_point_estimates"]}

if len(CAN_H3_CONTRASTS)!=32 or len(CAN_H3P_ROWS)!=150 or len(CAN_H4_ROWS)!=300:
    hard_stop("Canonical comparison universe changed")

def same_num(a,b,atol=1e-12):
    if a is None or b is None: return a is None and b is None
    return bool(np.isclose(float(a),float(b),rtol=0.0,atol=atol))

print("Canonical scientific inputs loaded; only derived source cluster_id changed in memory.",flush=True)


Canonical scientific inputs loaded; only derived source cluster_id changed in memory.


In [5]:
# 5. Canonical bootstrap/statistical primitives (same multiplicity law and fail-closed rules)
class ClusterBootstrapPlan:
    def __init__(self,cluster_ids,name):
        a=np.asarray(cluster_ids).astype(str)
        if len(a)==0: hard_stop("Cannot build bootstrap plan for empty frame",{"name":name})
        u=np.array(sorted(set(a)))
        code={x:i for i,x in enumerate(u)}
        self.row_cluster=np.asarray([code[x] for x in a],dtype=np.int32)
        self.cluster_labels=u;self.n_clusters=int(len(u));self.n_rows=int(len(a));self.name=str(name)
    def draw_row_counts(self,rng):
        sampled=rng.choice(self.n_clusters,size=self.n_clusters,replace=True)
        mult=np.bincount(sampled,minlength=self.n_clusters)
        return mult[self.row_cluster].astype(np.int64,copy=False)

def complete_ci(values):
    a=np.asarray(values,float)
    if len(a)!=BOOTSTRAP_REPS or not np.isfinite(a).all():
        raise ValueError("CI requires all 5,000 finite locked sensitivity replicates")
    return [float(np.percentile(a,2.5)),float(np.percentile(a,97.5))]

def complete_p(values):
    a=np.asarray(values,float)
    if len(a)!=BOOTSTRAP_REPS or not np.isfinite(a).all():
        raise ValueError("p-value requires all 5,000 finite locked sensitivity replicates")
    return float(min(1.0,2*min(np.mean(a<=0),np.mean(a>=0))))

def _weighted_mean(v,w):
    v=np.asarray(v,float);w=np.asarray(w,float);den=float(w.sum())
    return float(np.dot(v,w)/den) if den>0 else np.nan

def _two_class_reasons(y,w,prefix):
    y=np.asarray(y,int);w=np.asarray(w,float);out=[]
    if float(w[y==0].sum())<=0:out.append(f"{prefix}_missing_class_0")
    if float(w[y==1].sum())<=0:out.append(f"{prefix}_missing_class_1")
    return out

def _relative_change_failclosed(target,source,name):
    source=float(source);target=float(target)
    if not np.isfinite(source):return None,f"source_{name}_nonfinite"
    if source<=0:return None,f"source_{name}_nonpositive"
    if not np.isfinite(target):return None,f"target_{name}_nonfinite"
    v=(target-source)/source
    if not np.isfinite(v):return None,f"{name}_relative_change_nonfinite"
    return float(v),None

def _auc_setup(y,p):
    y=np.asarray(y,int);p=np.asarray(p,float);order=np.argsort(p,kind="mergesort");ps=p[order]
    starts=np.r_[0,np.flatnonzero(ps[1:]!=ps[:-1])+1]
    return order,starts

def _auc_from_row_counts(y,row_counts,setup):
    y=np.asarray(y,int);w=np.asarray(row_counts,float);order,starts=setup
    ys=y[order];ws=w[order];pos=np.add.reduceat(ws*ys,starts);neg=np.add.reduceat(ws*(1-ys),starts)
    P=float(pos.sum());N=float(neg.sum())
    if P<=0 or N<=0:return np.nan
    neg_before=np.cumsum(neg)-neg
    return float(np.sum(pos*(neg_before+0.5*neg))/(P*N))

def balanced_brier_point(y,p):
    y=np.asarray(y,int);p=np.asarray(p,float)
    if len(np.unique(y))<2:return None
    return 0.5*(float(np.mean((p[y==0]-0)**2))+float(np.mean((p[y==1]-1)**2)))

def deterministic_h1_h2_points(source,target):
    sy=source.label.to_numpy(int);ty=target.label.to_numpy(int)
    sp=np.clip(source.p_calibrated.to_numpy(float),1e-7,1-1e-7);tp=np.clip(target.p_calibrated.to_numpy(float),1e-7,1-1e-7)
    su=np.clip(source.p_uncalibrated.to_numpy(float),1e-7,1-1e-7);tu=np.clip(target.p_uncalibrated.to_numpy(float),1e-7,1-1e-7)
    sauc=float(roc_auc_score(sy,sp));tauc=float(roc_auc_score(ty,tp))
    snll=float(np.mean(-(sy*np.log(sp)+(1-sy)*np.log(1-sp))));tnll=float(np.mean(-(ty*np.log(tp)+(1-ty)*np.log(1-tp))))
    sbb=balanced_brier_point(sy,sp);tbb=balanced_brier_point(ty,tp)
    nllrel,_=_relative_change_failclosed(tnll,snll,"nll");bbrel,_=_relative_change_failclosed(tbb,sbb,"balanced_brier")
    sunll=float(np.mean(-(sy*np.log(su)+(1-sy)*np.log(1-su))));tunll=float(np.mean(-(ty*np.log(tu)+(1-ty)*np.log(1-tu))))
    gs=sunll-snll;gt=tunll-tnll
    return {"target_auroc":tauc,"auroc_drop":sauc-tauc,"nll_relative_change":nllrel,
            "balanced_brier_relative_change":bbrel,"source_nll_gain":gs,"target_nll_gain":gt,
            "target_minus_source_gain":gt-gs}

def simes_complete(pvals):
    if len(pvals)!=4 or not all(p is not None and np.isfinite(p) for p in pvals):return None
    p=np.sort(np.asarray(pvals,float))
    return float(min(1.0,np.min(p*len(p)/np.arange(1,len(p)+1))))

# H3 primitives
def item_frame(pred):
    return pred[(pred.model=="ebm")&(pred.seed.astype(int)==SEEDS[0])].drop_duplicates("row_id").sort_values("row_id",kind="stable").reset_index(drop=True)

def ebm_seed_frames(pred):
    out={};ids0=None
    for seed in SEEDS:
        g=pred[(pred.model=="ebm")&(pred.seed.astype(int)==seed)].sort_values("row_id",kind="stable").reset_index(drop=True)
        ids=g.row_id.astype(str).tolist()
        if ids0 is None:ids0=ids
        elif ids!=ids0:hard_stop("EBM seed row alignment changed",{"seed":seed})
        out[seed]=g
    return out

def h3_bundle(item,seed_frames,name):
    ids=item.row_id.astype(str).tolist();y=item.label.to_numpy(int)
    cues={c:(item[c].to_numpy()>0) for c in CUE_FAMILIES};losses=[]
    for seed in SEEDS:
        g=seed_frames[seed]
        if g.row_id.astype(str).tolist()!=ids:hard_stop("H3 row alignment changed",{"domain":name,"seed":seed})
        p=np.clip(g.p_calibrated.to_numpy(float),1e-7,1-1e-7)
        losses.append(-(y*np.log(p)+(1-y)*np.log(1-p)))
    return {"name":name,"row_ids":ids,"y":y,"cues":cues,"loss_by_seed":np.vstack(losses),
            "plan":ClusterBootstrapPlan(item.cluster_id.to_numpy(),name),"ones":np.ones(len(item),dtype=np.int16)}

def group_support_reasons(bundle,cue,row_counts,prefix):
    m=bundle["cues"][cue];w=np.asarray(row_counts);out=[]
    if float(w[m].sum())<=0:out.append(f"{prefix}_missing_cue_present")
    if float(w[~m].sum())<=0:out.append(f"{prefix}_missing_cue_absent")
    return out

def group_gap_1d(values,cue_present,row_counts):
    v=np.asarray(values,float);m=np.asarray(cue_present,bool);w=np.asarray(row_counts,float)
    dp=float(w[m].sum());da=float(w[~m].sum())
    if dp<=0 or da<=0:raise ValueError("undefined group gap")
    return float(np.dot(v[m],w[m])/dp-np.dot(v[~m],w[~m])/da)

def group_gap_by_seed(loss,cue_present,row_counts):
    loss=np.asarray(loss,float);m=np.asarray(cue_present,bool);w=np.asarray(row_counts,float)
    dp=float(w[m].sum());da=float(w[~m].sum())
    if dp<=0 or da<=0:raise ValueError("undefined seeded group gap")
    return (loss[:,m]@w[m])/dp-(loss[:,~m]@w[~m])/da

def support_record(bundle,cue):
    m=bundle["cues"][cue];rc=bundle["plan"].row_cluster;K=bundle["plan"].n_clusters
    kp=len(set(rc[m].tolist()));ka=len(set(rc[~m].tolist()))
    return {"rows":bundle["plan"].n_rows,"clusters":K,"cue_present_rows":int(m.sum()),"cue_absent_rows":int((~m).sum()),
            "clusters_with_cue_present":kp,"clusters_with_cue_absent":ka,
            "analytical_p_resample_missing_cue_present":float(((K-kp)/K)**K),
            "analytical_p_resample_missing_cue_absent":float(((K-ka)/K)**K)}

print("Canonical sensitivity statistical primitives loaded.",flush=True)


Canonical sensitivity statistical primitives loaded.


In [6]:
# 6. H1/H2 source-site-cluster sensitivity — same points, new source-side uncertainty only
domains={"source_test":SRC_SENS}
for p in PLATFORMS:domains[f"banglaclick:{p}"]=BC_P[BC_P.platform.eq(p)].copy()
domains["baitbuster:youtube_human"]=BB_P

plans={};row_ids={}
for d,df in domains.items():
    base=df[(df.model==MODEL_KEYS[0])&(df.seed.astype(int)==SEEDS[0])].sort_values("row_id",kind="stable").reset_index(drop=True)
    plans[d]=ClusterBootstrapPlan(base.cluster_id.to_numpy(),d);row_ids[d]=base.row_id.astype(str).tolist()

rng=np.random.default_rng(BOOTSTRAP_SEED)
sens_stat=[];sens_h1=[];sens_h2=[]
targets=[d for d in domains if d!="source_test"]

for dname in targets:
  for model in MODEL_KEYS:
    for seed in SEEDS:
        s=SRC_SENS[(SRC_SENS.model==model)&(SRC_SENS.seed==seed)].sort_values("row_id",kind="stable").reset_index(drop=True)
        t=domains[dname][(domains[dname].model==model)&(domains[dname].seed==seed)].sort_values("row_id",kind="stable").reset_index(drop=True)
        if s.row_id.astype(str).tolist()!=row_ids["source_test"] or t.row_id.astype(str).tolist()!=row_ids[dname]:
            hard_stop("H1/H2 row order differs from cluster plan")
        key=(dname,model,int(seed));can_h1=CAN_H1_ASSESS[key];can_h2=CAN_H2_ASSESS[key]
        pts=deterministic_h1_h2_points(s,t)
        for f in ["target_auroc","auroc_drop","nll_relative_change","balanced_brier_relative_change"]:
            if not same_num(pts[f],can_h1[f]):hard_stop("H1 point-estimate parity failed",{"setting":key,"field":f})
        for f in ["source_nll_gain","target_nll_gain","target_minus_source_gain"]:
            if not same_num(pts[f],can_h2[f]):hard_stop("H2 point-estimate parity failed",{"setting":key,"field":f})

        sy=s.label.to_numpy(int);ty=t.label.to_numpy(int)
        sp=np.clip(s.p_calibrated.to_numpy(float),1e-7,1-1e-7);tp=np.clip(t.p_calibrated.to_numpy(float),1e-7,1-1e-7)
        su=np.clip(s.p_uncalibrated.to_numpy(float),1e-7,1-1e-7);tu=np.clip(t.p_uncalibrated.to_numpy(float),1e-7,1-1e-7)
        sl=-(sy*np.log(sp)+(1-sy)*np.log(1-sp));tl=-(ty*np.log(tp)+(1-ty)*np.log(1-tp))
        sul=-(sy*np.log(su)+(1-sy)*np.log(1-su));tul=-(ty*np.log(tu)+(1-ty)*np.log(1-tu))
        ssq=(sp-sy)**2;tsq=(tp-ty)**2
        sas=_auc_setup(sy,sp);tas=_auc_setup(ty,tp)
        aucv=[];nllv=[];bbv=[];h2s=[];h2t=[];h2d=[]
        auc_reasons=Counter();nll_reasons=Counter();bb_reasons=Counter();overall_reasons=Counter();h2_reasons=Counter();overall_valid=0

        for rep in range(BOOTSTRAP_REPS):
            sw=plans["source_test"].draw_row_counts(rng);tw=plans[dname].draw_row_counts(rng)
            ar=_two_class_reasons(sy,sw,"source")+_two_class_reasons(ty,tw,"target");av=None
            if ar:
                for z in set(ar):auc_reasons[z]+=1
            else:
                a=_auc_from_row_counts(sy,sw,sas);b=_auc_from_row_counts(ty,tw,tas)
                if np.isfinite(a) and np.isfinite(b):av=float(a-b);aucv.append(av)
                else:auc_reasons["auroc_nonfinite"]+=1
            sn=_weighted_mean(sl,sw);tn=_weighted_mean(tl,tw);nv,nr=_relative_change_failclosed(tn,sn,"nll")
            if nr:nll_reasons[nr]+=1
            else:nllv.append(nv)
            br=_two_class_reasons(sy,sw,"source")+_two_class_reasons(ty,tw,"target");bv=None
            if br:
                for z in set(br):bb_reasons[z]+=1
            else:
                sbb=.5*(_weighted_mean(ssq[sy==0],sw[sy==0])+_weighted_mean(ssq[sy==1],sw[sy==1]))
                tbb=.5*(_weighted_mean(tsq[ty==0],tw[ty==0])+_weighted_mean(tsq[ty==1],tw[ty==1]))
                bv,rr=_relative_change_failclosed(tbb,sbb,"balanced_brier")
                if rr:bb_reasons[rr]+=1;bv=None
                else:bbv.append(bv)
            ors=[]
            if av is None:ors.append("auroc_invalid")
            if nv is None:ors.append(nr or "nll_invalid")
            if bv is None:ors.append("balanced_brier_invalid")
            if ors:overall_reasons["+".join(sorted(set(ors)))]+=1
            else:overall_valid+=1
            gs=_weighted_mean(sul,sw)-_weighted_mean(sl,sw);gt=_weighted_mean(tul,tw)-_weighted_mean(tl,tw)
            if np.isfinite([gs,gt,gt-gs]).all():h2s.append(gs);h2t.append(gt);h2d.append(gt-gs)
            else:h2_reasons["nonfinite_nll_gain"]+=1

        ae=len(aucv)==BOOTSTRAP_REPS and not auc_reasons
        ne=len(nllv)==BOOTSTRAP_REPS and not nll_reasons
        be=len(bbv)==BOOTSTRAP_REPS and not bb_reasons
        h1e=ae and ne and be and overall_valid==BOOTSTRAP_REPS
        h2e=len(h2d)==BOOTSTRAP_REPS and not h2_reasons
        st={
            "domain":dname,"model":model,"seed":int(seed),"bootstrap_attempts":BOOTSTRAP_REPS,
            "h1_valid_replicates":overall_valid,"h1_invalid_replicates":BOOTSTRAP_REPS-overall_valid,
            "h1_inferential_status":"ESTIMABLE" if h1e else "NON_ESTIMABLE_UNDER_SOURCE_SITE_CLUSTER_SENSITIVITY",
            "auroc_estimable":ae,"nll_relative_change_estimable":ne,"balanced_brier_relative_change_estimable":be,
            "auroc_valid_replicates":len(aucv),"nll_relative_change_valid_replicates":len(nllv),"balanced_brier_relative_change_valid_replicates":len(bbv),
            "auroc_invalid_reason_counts":dict(auc_reasons),"nll_relative_change_invalid_reason_counts":dict(nll_reasons),
            "balanced_brier_relative_change_invalid_reason_counts":dict(bb_reasons),"h1_overall_invalid_combination_counts":dict(overall_reasons),
            "auroc_drop_ci95":complete_ci(aucv) if ae else None,
            "nll_relative_change_ci95":complete_ci(nllv) if ne else None,
            "balanced_brier_relative_change_ci95":complete_ci(bbv) if be else None,
            "h2_valid_replicates":len(h2d),"h2_invalid_replicates":BOOTSTRAP_REPS-len(h2d),
            "h2_invalid_reason_counts":dict(h2_reasons),"h2_inferential_status":"ESTIMABLE" if h2e else "NON_ESTIMABLE_UNDER_SOURCE_SITE_CLUSTER_SENSITIVITY",
            "h2_source_gain_ci95":complete_ci(h2s) if h2e else None,
            "h2_target_gain_ci95":complete_ci(h2t) if h2e else None,
            "h2_target_minus_source_gain_ci95":complete_ci(h2d) if h2e else None,
            "h2_target_minus_source_gain_p":complete_p(h2d) if h2e else None,
        }
        sens_stat.append(st)

        cal_away=bool(can_h1["calibration_moved_further_from_ideal"])
        if not h1e:support1=None
        else:
            nll_ev=pts["nll_relative_change"]>=.10 and st["nll_relative_change_ci95"][0]>0
            bb_ev=pts["balanced_brier_relative_change"]>=.10 and st["balanced_brier_relative_change_ci95"][0]>0
            support1=bool(pts["target_auroc"]>=.70 and pts["auroc_drop"]<=.05 and (nll_ev or bb_ev) and cal_away)
        sens_h1.append({**pts,"domain":dname,"model":model,"seed":int(seed),"inferential_status":st["h1_inferential_status"],
                        "auroc_estimable":ae,"nll_relative_change_estimable":ne,"balanced_brier_relative_change_estimable":be,
                        "nll_bootstrap_ci95":st["nll_relative_change_ci95"],"balanced_brier_bootstrap_ci95":st["balanced_brier_relative_change_ci95"],
                        "calibration_moved_further_from_ideal":cal_away,"setting_supported":support1})
        if not h2e:support2=None
        else:
            support2=bool(pts["source_nll_gain"]>0 and st["h2_source_gain_ci95"][0]>0 and
                          pts["target_minus_source_gain"]<0 and st["h2_target_minus_source_gain_ci95"][1]<0)
        sens_h2.append({"domain":dname,"model":model,"seed":int(seed),"source_nll_gain":pts["source_nll_gain"],
                        "target_nll_gain":pts["target_nll_gain"],"target_minus_source_gain":pts["target_minus_source_gain"],
                        "inferential_status":st["h2_inferential_status"],"source_gain_ci95":st["h2_source_gain_ci95"],
                        "target_gain_ci95":st["h2_target_gain_ci95"],"target_minus_source_gain_ci95":st["h2_target_minus_source_gain_ci95"],
                        "p":st["h2_target_minus_source_gain_p"],"setting_supported":support2})

write_json(OUT_ROOT/"H1_H2_SOURCE_CLUSTER_SENSITIVITY.json",{
    "schema_version":1,"sensitivity_version":SENSITIVITY_VERSION,"cluster_rule_version":CLUSTER_RULE_VERSION,
    "bootstrap_replicates":BOOTSTRAP_REPS,"bootstrap_seed":BOOTSTRAP_SEED,
    "canonical_points_required_exact_parity":True,"h1_h2_bootstrap":sens_stat,
    "h1_confirmatory_assessment_rows":sens_h1,"h2_confirmatory_assessment_rows":sens_h2,
    "non_rescue_rule":"Sensitivity may never upgrade canonical non-estimability or canonical support.",
})
print("H1/H2 source-site cluster sensitivity complete.",flush=True)


H1/H2 source-site cluster sensitivity complete.


In [7]:
# 7. H3 fixed 32-cell + full 32-D + full eight-family multiplicity sensitivity; NO BaitBuster H3 replication
src_item=item_frame(SRC_SENS)
bc_items={p:item_frame(BC_P[BC_P.platform.eq(p)]) for p in PLATFORMS}
src_h3=h3_bundle(src_item,ebm_seed_frames(SRC_SENS),"source_test")
bc_h3={p:h3_bundle(bc_items[p],ebm_seed_frames(BC_P[BC_P.platform.eq(p)]),f"banglaclick:{p}") for p in PLATFORMS}

rng_h3=np.random.default_rng(BOOTSTRAP_SEED+1)
contrasts=[]
for cue in CUE_FAMILIES:
  for p in PLATFORMS:
    tgt=bc_h3[p];key=(cue,f"banglaclick:{p}");can=CAN_H3_CONTRASTS[key]
    point_reasons=group_support_reasons(src_h3,cue,src_h3["ones"],"source")+group_support_reasons(tgt,cue,tgt["ones"],"target")
    if point_reasons:
        point_rd=point_nll=None;by_seed=[None]*5
    else:
        srd=group_gap_1d(src_h3["y"],src_h3["cues"][cue],src_h3["ones"]);trd=group_gap_1d(tgt["y"],tgt["cues"][cue],tgt["ones"])
        point_rd=float(trd-srd)
        sn=group_gap_by_seed(src_h3["loss_by_seed"],src_h3["cues"][cue],src_h3["ones"])
        tn=group_gap_by_seed(tgt["loss_by_seed"],tgt["cues"][cue],tgt["ones"]);d=tn-sn
        point_nll=float(np.mean(d));by_seed=[float(x) for x in d]
    if not same_num(point_rd,can["delta_rd"]) or not same_num(point_nll,can["ebm_delta_nll_gap_mean_5seeds"]):
        hard_stop("H3 point-estimate parity failed",{"cue":cue,"platform":p})
    rb=[];nb=[];reasons=Counter()
    for rep in range(BOOTSTRAP_REPS):
        sw=src_h3["plan"].draw_row_counts(rng_h3);tw=tgt["plan"].draw_row_counts(rng_h3)
        rr=group_support_reasons(src_h3,cue,sw,"source")+group_support_reasons(tgt,cue,tw,"target")
        if rr:reasons["+".join(sorted(rr))]+=1
        else:
            rb.append(group_gap_1d(tgt["y"],tgt["cues"][cue],tw)-group_gap_1d(src_h3["y"],src_h3["cues"][cue],sw))
            sd=group_gap_by_seed(src_h3["loss_by_seed"],src_h3["cues"][cue],sw);td=group_gap_by_seed(tgt["loss_by_seed"],tgt["cues"][cue],tw)
            nb.append(float(np.mean(td-sd)))
    invalid=sum(reasons.values());estimable=(not point_reasons) and invalid==0
    contrasts.append({
        "cue":cue,"target":f"banglaclick:{p}","delta_rd":point_rd,
        "delta_rd_ci95":complete_ci(rb) if estimable else None,"delta_rd_p":complete_p(rb) if estimable else None,
        "ebm_delta_nll_gap_mean_5seeds":point_nll,"ebm_delta_nll_gap_ci95":complete_ci(nb) if estimable else None,
        "ebm_delta_nll_gap_p":complete_p(nb) if estimable else None,"ebm_delta_nll_gap_by_seed":by_seed,
        "bootstrap_attempts":BOOTSTRAP_REPS,"valid_replicates":BOOTSTRAP_REPS-invalid,"invalid_replicates":invalid,
        "invalid_reason_counts":dict(reasons),"inferential_status":"ESTIMABLE" if estimable else "NON_ESTIMABLE_UNDER_SOURCE_SITE_CLUSTER_SENSITIVITY",
        "source_support":support_record(src_h3,cue),"target_support":support_record(tgt,cue),
    })

def family_simes(endpoint):
    out={}
    for cue in CUE_FAMILIES:
        rr=[r for r in contrasts if r["cue"]==cue];ps=[r[endpoint] for r in rr];sp=simes_complete(ps)
        out[cue]={"status":"ESTIMABLE" if sp is not None else "NON_ESTIMABLE_INCOMPLETE_FOUR_PLATFORM_FAMILY",
                  "simes_p":sp,"component_p_values":ps,
                  "nonestimable_platforms":[r["target"] for r in rr if r["inferential_status"]!="ESTIMABLE"]}
    return out
fam_out=family_simes("delta_rd_p");fam_rel=family_simes("ebm_delta_nll_gap_p")

def full_bh(fam):
    ps=[fam[c]["simes_p"] for c in CUE_FAMILIES]
    if not all(p is not None and np.isfinite(p) for p in ps):
        return {"status":"NON_ESTIMABLE_FULL_8_FAMILY_BH","q":FDR_Q,"number_prespecified_families":8,
                "number_estimable_family_p_values":sum(p is not None for p in ps),
                "rows":{c:{"simes_p":fam[c]["simes_p"],"bh_q":None,"reject_q_0_05":None,
                           "status":"BH_NOT_RUN_FULL_PRESPECIFIED_FAMILY_INCOMPLETE"} for c in CUE_FAMILIES}}
    reject,qvals,_,_=multipletests(ps,alpha=FDR_Q,method="fdr_bh")
    return {"status":"ESTIMABLE","q":FDR_Q,"number_prespecified_families":8,"number_estimable_family_p_values":8,
            "rows":{c:{"simes_p":float(p),"bh_q":float(q),"reject_q_0_05":bool(r),"status":"ESTIMABLE"}
                    for c,p,q,r in zip(CUE_FAMILIES,ps,qvals,reject)}}
fdr_out=full_bh(fam_out);fdr_rel=full_bh(fam_rel)

# Full fixed 32-D global attempt; no dimension deletion.
point_vec=np.asarray([CAN_H3_CONTRASTS[(c,f"banglaclick:{p}")]["delta_rd"] for c in CUE_FAMILIES for p in PLATFORMS],float)
B=np.full((BOOTSTRAP_REPS,32),np.nan);reason_counts=[Counter() for _ in range(32)]
rng_g=np.random.default_rng(BOOTSTRAP_SEED+2)
for rep in range(BOOTSTRAP_REPS):
    sw=src_h3["plan"].draw_row_counts(rng_g);tw={p:bc_h3[p]["plan"].draw_row_counts(rng_g) for p in PLATFORMS};j=0
    for cue in CUE_FAMILIES:
        sr=group_support_reasons(src_h3,cue,sw,"source")
        for p in PLATFORMS:
            rr=sr+group_support_reasons(bc_h3[p],cue,tw[p],"target")
            if rr:reason_counts[j]["+".join(sorted(rr))]+=1
            else:B[rep,j]=group_gap_1d(bc_h3[p]["y"],bc_h3[p]["cues"][cue],tw[p])-group_gap_1d(src_h3["y"],src_h3["cues"][cue],sw)
            j+=1
dimension_records=[]
j=0
for cue in CUE_FAMILIES:
    for p in PLATFORMS:
        invalid_count=int((~np.isfinite(B[:,j])).sum())
        dimension_records.append({
            "dimension_index":j,"cue":cue,"platform":p,
            "valid_replicates":int(BOOTSTRAP_REPS-invalid_count),
            "invalid_replicates":invalid_count,
            "invalid_rate":float(invalid_count/BOOTSTRAP_REPS),
            "invalid_reason_counts":dict(sorted(reason_counts[j].items())),
        })
        j+=1
joint=np.isfinite(B).all(axis=1)
global_res={"status":None,"expected_dimension":32,"dimension_reduction_performed":False,"bootstrap_attempts":BOOTSTRAP_REPS,
            "joint_valid_replicates":int(joint.sum()),"joint_invalid_replicates":int((~joint).sum()),
            "dimension_estimability":dimension_records,
            "statistic":None,"df":None,"p":None,"covariance_rank":None}
if not np.isfinite(point_vec).all() or not joint.all():
    global_res["status"]="NON_ESTIMABLE"
else:
    cov=np.atleast_2d(np.cov(B,rowvar=False));rank=int(np.linalg.matrix_rank(cov));global_res["covariance_rank"]=rank
    if rank!=32:global_res["status"]="NON_ESTIMABLE"
    else:
        try:wald=float(point_vec@np.linalg.solve(cov,point_vec))
        except np.linalg.LinAlgError:global_res["status"]="NON_ESTIMABLE"
        else:global_res.update({"status":"ESTIMABLE","statistic":wald,"df":32,"p":float(chi2.sf(wald,32))})

write_json(OUT_ROOT/"H3_SOURCE_CLUSTER_SENSITIVITY.json",{
    "schema_version":1,"sensitivity_version":SENSITIVITY_VERSION,"cluster_rule_version":CLUSTER_RULE_VERSION,
    "scope":"BanglaClick fixed 32-cell H3 only; canonical BaitBuster H3 replication is not rerun in Stage09.",
    "bootstrap_replicates":BOOTSTRAP_REPS,"bootstrap_seed":BOOTSTRAP_SEED,
    "banglaclick_cue_platform_contrasts":contrasts,
    "global_banglaclick_outcome_wald":global_res,
    "family_simes_outcome":fam_out,"family_simes_reliability":fam_rel,
    "family_fdr_outcome_full":fdr_out,"family_fdr_reliability_full":fdr_rel,
    "non_rescue_rule":"Sensitivity estimability never upgrades canonical H3 status.",
    "reduced_24d_test_performed":False,"subset_bh_performed":False,
})
print("H3 source-site cluster sensitivity complete; fixed 32-D universe preserved.",flush=True)


H3 source-site cluster sensitivity complete; fixed 32-D universe preserved.


In [8]:
# 8. H3-P source-site-cluster sensitivity from immutable pair cache
source_groups={(cue,model,int(seed)):g.sort_values("row_id",kind="stable").reset_index(drop=True)
               for (cue,model,seed),g in PERT_SENS[PERT_SENS.domain.eq("source_test")].groupby(["cue","model","seed"],sort=True)}
targets=[]
for (cue,model,seed),sg in PERT_SENS[PERT_SENS.domain.eq("source_test")].groupby(["cue","model","seed"],sort=True):
    q=PERT_SENS[(~PERT_SENS.domain.eq("source_test"))&(PERT_SENS.cue.eq(cue))&(PERT_SENS.model.eq(model))&(PERT_SENS.seed.eq(seed))]
    for dname,tg in q.groupby("domain",sort=True):targets.append((cue,model,int(seed),dname,tg.sort_values("row_id",kind="stable").reset_index(drop=True)))
if len(targets)!=150:hard_stop("H3-P frozen contrast universe changed",{"observed":len(targets)})

plan_cache={}
def pplan(domain,cue,g):
    key=(domain,cue,tuple(g.row_id.astype(str).tolist()))
    if key not in plan_cache:plan_cache[key]=ClusterBootstrapPlan(g.cluster_id.to_numpy(),f"{domain}|{cue}")
    return plan_cache[key]

rng=np.random.default_rng(BOOTSTRAP_SEED+4);rows=[]
for cue,model,seed,dname,tg in targets:
    sg=source_groups[(cue,model,seed)];key=(cue,model,seed,dname);can=CAN_H3P_ROWS[key]
    ps=float(sg.delta_z.mean());pt=float(tg.delta_z.mean());pdiff=pt-ps
    if not (same_num(ps,can["source_mean_delta_z"]) and same_num(pt,can["target_mean_delta_z"]) and same_num(pdiff,can["target_minus_source_delta_z"])):
        hard_stop("H3-P point-estimate parity failed",{"contrast":key})
    spl=pplan("source_test",cue,sg);tpl=pplan(dname,cue,tg);sv=sg.delta_z.to_numpy(float);tv=tg.delta_z.to_numpy(float)
    vals=[];invalid_reasons=Counter()
    for rep in range(BOOTSTRAP_REPS):
        sw=spl.draw_row_counts(rng);tw=tpl.draw_row_counts(rng);v=_weighted_mean(tv,tw)-_weighted_mean(sv,sw)
        if np.isfinite(v):vals.append(float(v))
        else:invalid_reasons["nonfinite_delta_z_difference"]+=1
    invalid=int(sum(invalid_reasons.values()))
    est=invalid==0 and len(vals)==BOOTSTRAP_REPS
    rows.append({"cue":cue,"model":model,"seed":seed,"target":dname,"source_mean_delta_z":ps,"target_mean_delta_z":pt,
                 "target_minus_source_delta_z":pdiff,"bootstrap_attempts":BOOTSTRAP_REPS,"valid_replicates":len(vals),"invalid_replicates":invalid,
                 "invalid_reason_counts":dict(sorted(invalid_reasons.items())),
                 "inferential_status":"ESTIMABLE" if est else "NON_ESTIMABLE_UNDER_SOURCE_SITE_CLUSTER_SENSITIVITY",
                 "ci95":complete_ci(vals) if est else None,"p":complete_p(vals) if est else None})
write_json(OUT_ROOT/"H3P_SOURCE_CLUSTER_SENSITIVITY.json",{
    "schema_version":1,"sensitivity_version":SENSITIVITY_VERSION,"cluster_rule_version":CLUSTER_RULE_VERSION,
    "response":"delta_z = original logit - cue-neutralized logit","causal_real_world_claim":False,
    "models":["ebm","banglabert"],"families":PERT_FAMS,"rows":rows,
    "non_rescue_rule":"Sensitivity may not upgrade the canonical H3-P claim.",
})
print("H3-P source-site cluster sensitivity complete.",flush=True)


H3-P source-site cluster sensitivity complete.


In [9]:
# 9. H4 source-site-cluster sensitivity — same canonical policy/points, 5/10/20% with 10% primary
def uncertainty_score(p):return 1-np.maximum(p,1-p)
_HARMONIC=np.array([0.0])
def _ensure_harmonic(n):
    global _HARMONIC
    if n<len(_HARMONIC):return
    x=np.arange(len(_HARMONIC),n+1,dtype=float);_HARMONIC=np.concatenate([_HARMONIC,_HARMONIC[-1]+np.cumsum(1/x)])
def _weighted_prefix(cw,cv,v,take):
    j=int(np.searchsorted(cw,int(take),side="left"));pw=float(cw[j-1]) if j else 0.;pv=float(cv[j-1]) if j else 0.
    return pv+(take-pw)*float(v[j])
class OperationalPlan:
    def __init__(self,g,name):
        f=g.sort_values("row_id",kind="stable").reset_index(drop=True);self.name=name;self.n=len(f)
        self.row_id=f.row_id.astype(str).to_numpy();self.y=f.label.astype(int).to_numpy();self.p=f.p_calibrated.astype(float).to_numpy()
        self.error=((self.p>=.5).astype(int)!=self.y).astype(np.int8);self.u=uncertainty_score(self.p)
        tmp=pd.DataFrame({"i":np.arange(self.n),"row_id":self.row_id,"p":self.p,"u":self.u})
        self.pos=tmp.sort_values(["p","row_id"],ascending=[False,True],kind="stable").i.to_numpy(int)
        self.uh=tmp.sort_values(["u","row_id"],ascending=[False,True],kind="stable").i.to_numpy(int)
        self.ul=tmp.sort_values(["u","row_id"],ascending=[True,True],kind="stable").i.to_numpy(int)
        self.cluster=ClusterBootstrapPlan(f.cluster_id.to_numpy(),name);self.ones=np.ones(self.n,dtype=np.int64)
    def metrics(self,w):
        w=np.asarray(w,np.int64);n=int(w.sum());tp=float(np.dot(self.y,w));te=float(np.dot(self.error,w));prev=tp/n
        pw=w[self.pos];pcw=np.cumsum(pw);pcy=np.cumsum(pw*self.y[self.pos])
        uw=w[self.uh];ucw=np.cumsum(uw);uce=np.cumsum(uw*self.error[self.uh]);bout={}
        for k in BUDGETS:
            take=max(1,int(np.ceil(k*n)));cp=_weighted_prefix(pcw,pcy,self.y[self.pos],take);ce=_weighted_prefix(ucw,uce,self.error[self.uh],take)
            precision=cp/take;lift=precision/prev if tp>0 else np.nan;ec=ce/te if te>0 else np.nan
            bout[float(k)]={"lift":float(lift) if np.isfinite(lift) else np.nan,"error_capture":float(ec) if np.isfinite(ec) else np.nan}
        lw=w[self.ul];le=self.error[self.ul].astype(np.int64);cw=np.cumsum(lw);before=cw-lw;prior=np.cumsum(lw*le)-lw*le
        _ensure_harmonic(n);dH=_HARMONIC[cw]-_HARMONIC[before]
        sr=float(np.where(le==0,prior*dH,lw+(prior-before)*dH).sum());fi=int(np.argmax(lw>0));fr=float(le[fi]);lr=float(te/n)
        aurc=fr if n==1 else float((sr-.5*fr-.5*lr)/n)
        return bout,aurc,{"total_pos":tp,"total_err":te}

h4domains={"source_test":SRC_SENS,"baitbuster:youtube_human":BB_P}
for p in PLATFORMS:h4domains[f"banglaclick:{p}"]=BC_P[BC_P.platform.eq(p)]
plans4={};points4={}
for d,df in h4domains.items():
    for (m,s),g in df.groupby(["model","seed"],sort=True):
        pl=OperationalPlan(g,f"{d}|{m}|{s}");plans4[(d,m,int(s))]=pl;points4[(d,m,int(s))]=pl.metrics(pl.ones)

# Exact canonical point parity across all 360 domain×model×seed×budget rows.
for (d,m,s),pl in plans4.items():
    bud,aurc,state=points4[(d,m,s)]
    for k in BUDGETS:
        can=CAN_H4_POINTS[(d,m,s,float(k))]
        if not same_num(bud[float(k)]["lift"],can["lift"]) or not same_num(bud[float(k)]["error_capture"],can["error_capture"]) or not same_num(aurc,can["aurc"]):
            hard_stop("H4 point-estimate parity failed",{"domain":d,"model":m,"seed":s,"budget":k})

def fin(values,reasons):
    a=np.asarray(values,float);valid=int(np.isfinite(a).sum());invalid=BOOTSTRAP_REPS-valid
    if len(a)!=BOOTSTRAP_REPS or sum(reasons.values())!=invalid:hard_stop("H4 invalid-reason accounting failure")
    est=invalid==0
    return {"bootstrap_attempts":BOOTSTRAP_REPS,"valid_replicates":valid,"invalid_replicates":invalid,
            "invalid_reason_counts":dict(reasons),"estimability_status":"ESTIMABLE" if est else "NON_ESTIMABLE_UNDER_SOURCE_SITE_CLUSTER_SENSITIVITY",
            "ci95":complete_ci(a) if est else None,"p_two_sided":complete_p(a) if est else None}

rng=np.random.default_rng(BOOTSTRAP_SEED);out=[]
for d in [x for x in h4domains if x!="source_test"]:
  for m in MODEL_KEYS:
    for seed in SEEDS:
        sp=plans4[("source_test",m,seed)];tp=plans4[(d,m,seed)];sb,sa,ss=points4[("source_test",m,seed)];tb,ta,ts=points4[(d,m,seed)]
        stores={float(k):{"lift":[],"ec":[],"lr":Counter(),"er":Counter()} for k in BUDGETS};av=[];ar=Counter()
        for rep in range(BOOTSTRAP_REPS):
            sw=sp.cluster.draw_row_counts(rng);tw=tp.cluster.draw_row_counts(rng);sbb,saa,sss=sp.metrics(sw);tbb,taa,tss=tp.metrics(tw)
            for k in BUDGETS:
                kk=float(k);lr=[]
                if sss["total_pos"]<=0:lr.append("source_total_pos_zero")
                if tss["total_pos"]<=0:lr.append("target_total_pos_zero")
                if lr:
                    stores[kk]["lift"].append(np.nan);stores[kk]["lr"]["+".join(sorted(lr))]+=1
                else:
                    dv=float(tbb[kk]["lift"]-sbb[kk]["lift"])
                    if np.isfinite(dv):stores[kk]["lift"].append(dv)
                    else:stores[kk]["lift"].append(np.nan);stores[kk]["lr"]["other_nonfinite_lift_endpoint"]+=1
                er=[]
                if sss["total_err"]<=0:er.append("source_total_err_zero")
                if tss["total_err"]<=0:er.append("target_total_err_zero")
                if er:
                    stores[kk]["ec"].append(np.nan);stores[kk]["er"]["+".join(sorted(er))]+=1
                else:
                    dv=float(tbb[kk]["error_capture"]-sbb[kk]["error_capture"])
                    if np.isfinite(dv):stores[kk]["ec"].append(dv)
                    else:stores[kk]["ec"].append(np.nan);stores[kk]["er"]["other_nonfinite_error_capture_endpoint"]+=1
            dv=taa-saa
            if np.isfinite(dv):av.append(float(dv))
            else:av.append(np.nan);ar["other_nonfinite_aurc_endpoint"]+=1
        af=fin(av,ar)
        for k in BUDGETS:
            kk=float(k);lf=fin(stores[kk]["lift"],stores[kk]["lr"]);ef=fin(stores[kk]["ec"],stores[kk]["er"])
            can=CAN_H4_ROWS[(d,m,seed,kk)]
            out.append({"domain":d,"model":m,"seed":seed,"budget":kk,"primary_budget":kk==PRIMARY_BUDGET,
                        "target_minus_source_lift":can["target_minus_source_lift"],"lift_ci95":lf["ci95"],"lift_p":lf["p_two_sided"],
                        "lift_estimability_status":lf["estimability_status"],"lift_valid_replicates":lf["valid_replicates"],"lift_invalid_replicates":lf["invalid_replicates"],"lift_invalid_reason_counts":lf["invalid_reason_counts"],
                        "target_minus_source_error_capture":can["target_minus_source_error_capture"],"error_capture_ci95":ef["ci95"],"error_capture_p":ef["p_two_sided"],
                        "error_capture_estimability_status":ef["estimability_status"],"error_capture_valid_replicates":ef["valid_replicates"],"error_capture_invalid_replicates":ef["invalid_replicates"],"error_capture_invalid_reason_counts":ef["invalid_reason_counts"],
                        "target_minus_source_aurc":can["target_minus_source_aurc"],"aurc_ci95":af["ci95"],"aurc_p":af["p_two_sided"],
                        "aurc_estimability_status":af["estimability_status"],"aurc_valid_replicates":af["valid_replicates"],"aurc_invalid_replicates":af["invalid_replicates"],"aurc_invalid_reason_counts":af["invalid_reason_counts"]})
write_json(OUT_ROOT/"H4_SOURCE_CLUSTER_SENSITIVITY.json",{
    "schema_version":1,"sensitivity_version":SENSITIVITY_VERSION,"cluster_rule_version":CLUSTER_RULE_VERSION,
    "budgets":BUDGETS,"primary_budget":PRIMARY_BUDGET,"bootstrap_replicates":BOOTSTRAP_REPS,"bootstrap_seed":BOOTSTRAP_SEED,
    "rows":out,"non_rescue_rule":"Sensitivity never replaces or upgrades canonical Stage-06 H4 inference.",
})
print("H4 source-site cluster sensitivity complete.",flush=True)


H4 source-site cluster sensitivity complete.


In [10]:
# 10. Exact canonical-vs-sensitivity concordance table; never upgrade canonical results
S_H12=json.loads((OUT_ROOT/"H1_H2_SOURCE_CLUSTER_SENSITIVITY.json").read_text(encoding="utf-8"))
S_H3=json.loads((OUT_ROOT/"H3_SOURCE_CLUSTER_SENSITIVITY.json").read_text(encoding="utf-8"))
S_H3P=json.loads((OUT_ROOT/"H3P_SOURCE_CLUSTER_SENSITIVITY.json").read_text(encoding="utf-8"))
S_H4=json.loads((OUT_ROOT/"H4_SOURCE_CLUSTER_SENSITIVITY.json").read_text(encoding="utf-8"))
SSTAT={(r["domain"],r["model"],int(r["seed"])):r for r in S_H12["h1_h2_bootstrap"]}
SH1={(r["domain"],r["model"],int(r["seed"])):r for r in S_H12["h1_confirmatory_assessment_rows"]}
SH2={(r["domain"],r["model"],int(r["seed"])):r for r in S_H12["h2_confirmatory_assessment_rows"]}
SH3={(r["cue"],r["target"]):r for r in S_H3["banglaclick_cue_platform_contrasts"]}
SH3P={(r["cue"],r["model"],int(r["seed"]),r["target"]):r for r in S_H3P["rows"]}
SH4={(r["domain"],r["model"],int(r["seed"]),float(r["budget"])):r for r in S_H4["rows"]}

def ci_width(ci):
    return None if not isinstance(ci,list) or len(ci)!=2 else float(ci[1]-ci[0])

def classify(cstat,sstat,cp,sp,cci,sci,csupport=None,ssupport=None,note=""):
    if cp is not None and sp is not None and float(cp)!=0 and float(sp)!=0 and np.sign(float(cp))!=np.sign(float(sp)):
        return "DIRECTION_REVERSAL","point direction reversed; inspect mapping"
    if csupport is True and ssupport is not True:
        return "LOCKED_SUPPORT_NO_LONGER_MET","canonical locked support is not retained under source-site clustering"
    ce=(cstat=="ESTIMABLE");se=(sstat=="ESTIMABLE")
    if ce and not se:
        return "UNCERTAINTY_WIDENED_INTERPRETATION_UNCHANGED","sensitivity becomes non-estimable; canonical claim remains authoritative"
    cw=ci_width(cci);sw=ci_width(sci)
    if ce and se and cw is not None and sw is not None and sw>cw+1e-12:
        return "UNCERTAINTY_WIDENED_INTERPRETATION_UNCHANGED","sensitivity CI is wider; interpretation otherwise unchanged"
    if not ce and se:
        return "CONCORDANT","sensitivity is estimable but canonical non-estimability is not upgraded"
    return "CONCORDANT",note or "canonical interpretation unchanged"

rows=[]
def add(claim,cstat,sstat,cp,sp,cci,sci,cpv,spv,csup=None,ssup=None,notes=""):
    if cp is not None and sp is not None and not same_num(cp,sp):hard_stop("Sensitivity changed a canonical point estimate",{"claim":claim})
    cls,n2=classify(cstat,sstat,cp,sp,cci,sci,csup,ssup,notes)
    rows.append({"claim":claim,"canonical_status":cstat,"sensitivity_status":sstat,
                 "canonical_point_estimate":cp,"sensitivity_point_estimate":sp,
                 "canonical_CI":json.dumps(cci) if cci is not None else None,"sensitivity_CI":json.dumps(sci) if sci is not None else None,
                 "canonical_p":cpv,"sensitivity_p":spv,
                 "canonical_estimability":cstat=="ESTIMABLE","sensitivity_estimability":sstat=="ESTIMABLE",
                 "interpretation_class":cls,"notes":n2})

# H1 endpoint rows + locked setting-support row.
for key,can in CAN_H1_ASSESS.items():
    ss=SH1[key];cs=CAN_STAT_ROWS[key];st=SSTAT[key];d,m,seed=key
    for name,point,cf,sf,ce,se in [
        ("AUROC_DROP",can["auroc_drop"],cs["auroc_drop_ci95"],st["auroc_drop_ci95"],cs["auroc_estimable"],st["auroc_estimable"]),
        ("NLL_RELATIVE_CHANGE",can["nll_relative_change"],cs["nll_relative_change_ci95"],st["nll_relative_change_ci95"],cs["nll_relative_change_estimable"],st["nll_relative_change_estimable"]),
        ("BALANCED_BRIER_RELATIVE_CHANGE",can["balanced_brier_relative_change"],cs["balanced_brier_relative_change_ci95"],st["balanced_brier_relative_change_ci95"],cs["balanced_brier_relative_change_estimable"],st["balanced_brier_relative_change_estimable"]),
    ]:
        add(f"H1|{name}|{d}|{m}|{seed}","ESTIMABLE" if ce else "NON_ESTIMABLE", "ESTIMABLE" if se else "NON_ESTIMABLE",
            point,point,cf,sf,None,None)
    add(f"H1|LOCKED_SETTING_SUPPORT|{d}|{m}|{seed}",can["inferential_status"],
        "ESTIMABLE" if ss["inferential_status"]=="ESTIMABLE" else "NON_ESTIMABLE",
        None,None,None,None,None,None,can["setting_supported"],ss["setting_supported"])

# H2
for key,can in CAN_H2_ASSESS.items():
    ss=SH2[key];d,m,seed=key
    add(f"H2|GAIN_ATTENUATION|{d}|{m}|{seed}",can["inferential_status"],
        "ESTIMABLE" if ss["inferential_status"]=="ESTIMABLE" else "NON_ESTIMABLE",
        can["target_minus_source_gain"],ss["target_minus_source_gain"],can["target_minus_source_gain_ci95"],ss["target_minus_source_gain_ci95"],
        CAN_STAT_ROWS[key]["h2_target_minus_source_gain_p"],ss["p"],can["setting_supported"],ss["setting_supported"])

# H3 individual endpoints
for key,can in CAN_H3_CONTRASTS.items():
    ss=SH3[key];cue,target=key
    for ep,point,cci,sci,cpv,spv in [
        ("DELTA_RD",can["delta_rd"],can["delta_rd_ci95"],ss["delta_rd_ci95"],can["delta_rd_p"],ss["delta_rd_p"]),
        ("DELTA_NLL_GAP",can["ebm_delta_nll_gap_mean_5seeds"],can["ebm_delta_nll_gap_ci95"],ss["ebm_delta_nll_gap_ci95"],can["ebm_delta_nll_gap_p"],ss["ebm_delta_nll_gap_p"]),
    ]:
        add(f"H3|{ep}|{cue}|{target}",can["inferential_status"],
            "ESTIMABLE" if ss["inferential_status"]=="ESTIMABLE" else "NON_ESTIMABLE",
            point,point,cci,sci,cpv,spv)

# Global H3 status
add("H3|GLOBAL_32D_WALD",CAN_H3["global_banglaclick_outcome_wald"]["status"],S_H3["global_banglaclick_outcome_wald"]["status"],
    None,None,None,None,CAN_H3["global_banglaclick_outcome_wald"].get("p"),S_H3["global_banglaclick_outcome_wald"].get("p"))

# Full eight-family H3 multiplicity procedures (procedure-level concordance; never subset-rescued).
for endpoint,can_full,sens_full in [
    ("OUTCOME",CAN_H3["family_fdr_outcome_full"],S_H3["family_fdr_outcome_full"]),
    ("RELIABILITY",CAN_H3["family_fdr_reliability_full"],S_H3["family_fdr_reliability_full"]),
]:
    cstat="ESTIMABLE" if can_full["status"]=="ESTIMABLE" else "NON_ESTIMABLE"
    sstat="ESTIMABLE" if sens_full["status"]=="ESTIMABLE" else "NON_ESTIMABLE"
    add(f"H3|FULL_8_FAMILY_BH|{endpoint}",cstat,sstat,None,None,None,None,None,None,
        None,None,
        f"canonical={can_full['status']}; sensitivity={sens_full['status']}; no post-hoc subset BH")

# H3-P
for key,can in CAN_H3P_ROWS.items():
    ss=SH3P[key];cue,m,seed,target=key
    add(f"H3P|DELTA_Z|{cue}|{m}|{seed}|{target}",can["inferential_status"],
        "ESTIMABLE" if ss["inferential_status"]=="ESTIMABLE" else "NON_ESTIMABLE",
        can["target_minus_source_delta_z"],ss["target_minus_source_delta_z"],can["ci95"],ss["ci95"],can["p"],ss["p"])

# H4
for key,can in CAN_H4_ROWS.items():
    ss=SH4[key];d,m,seed,budget=key
    for ep,point,cf,sf,cpv,spv,degdir in [
        ("LIFT",can["target_minus_source_lift"],can["lift_ci95"],ss["lift_ci95"],can["lift_p"],ss["lift_p"],lambda x:x<0),
        ("ERROR_CAPTURE",can["target_minus_source_error_capture"],can["error_capture_ci95"],ss["error_capture_ci95"],can["error_capture_p"],ss["error_capture_p"],lambda x:x<0),
        ("AURC",can["target_minus_source_aurc"],can["aurc_ci95"],ss["aurc_ci95"],can["aurc_p"],ss["aurc_p"],lambda x:x>0),
    ]:
        cstat=can[ep.lower()+"_estimability_status"] if ep!="ERROR_CAPTURE" else can["error_capture_estimability_status"]
        sstat=ss[ep.lower()+"_estimability_status"] if ep!="ERROR_CAPTURE" else ss["error_capture_estimability_status"]
        csup=bool(cstat=="ESTIMABLE" and cpv is not None and cpv<.05 and point is not None and degdir(point))
        ssup=bool(sstat=="ESTIMABLE" and spv is not None and spv<.05 and point is not None and degdir(point))
        add(f"H4|{ep}|{d}|{m}|{seed}|{budget}",cstat,"ESTIMABLE" if sstat=="ESTIMABLE" else "NON_ESTIMABLE",
            point,point,cf,sf,cpv,spv,csup,ssup)

CONC=pd.DataFrame(rows)
CONC.to_csv(OUT_ROOT/"CLAIM_CONCORDANCE_TABLE.csv",index=False)
print("Claim concordance table written:",len(CONC),"rows",flush=True)


Claim concordance table written: 1617 rows


In [11]:
# 11. Sensitivity summary, manifest/checksum contract, and PASS
CONC=pd.read_csv(OUT_ROOT/"CLAIM_CONCORDANCE_TABLE.csv")
classes=CONC["interpretation_class"].value_counts().to_dict()
diag=json.loads((OUT_ROOT/"SOURCE_CLUSTER_DIAGNOSTICS.json").read_text(encoding="utf-8"))
h3s=json.loads((OUT_ROOT/"H3_SOURCE_CLUSTER_SENSITIVITY.json").read_text(encoding="utf-8"))

summary=[
    "# Stage 09 source-site cluster sensitivity",
    "",
    f"- Sensitivity role: source-side dependence robustness only; canonical Stages 05–08 remain unchanged.",
    f"- Cluster rule: `{CLUSTER_RULE_VERSION}`.",
    f"- Source TEST rows mapped: {diag['total_source_test_rows']}.",
    f"- Source-site host clusters: {diag['total_publisher_clusters']}.",
    f"- Singleton clusters: {diag['singleton_clusters']}.",
    f"- Largest cluster share: {diag['largest_cluster_share']:.4f}.",
    f"- Top-5 / top-10 cluster share: {diag['top5_cluster_share']:.4f} / {diag['top10_cluster_share']:.4f}.",
    f"- Repeated-cluster observation proportion: {diag['proportion_observations_in_repeated_clusters']:.4f}.",
    f"- Few-cluster caution (<{diag['few_cluster_warning_threshold']}): {diag['few_cluster_warning']}.",
    f"- H3 sensitivity global 32-D status: {h3s['global_banglaclick_outcome_wald']['status']}.",
    f"- Concordance classifications: {json.dumps(classes,sort_keys=True)}.",
    "- Canonical non-estimable results are never upgraded by this sensitivity.",
    "- No Stage-07 bridge or LOO analysis was run.",
    "- No model training, calibration fitting, prediction generation, cue redefinition, target tuning, reduced H3, or subset BH occurred.",
]
write_text(OUT_ROOT/"SENSITIVITY_SUMMARY.md","\n".join(summary)+"\n")

required_scientific=[
    "SOURCE_PUBLISHER_MAPPING.csv","SOURCE_CLUSTER_DIAGNOSTICS.json",
    "H1_H2_SOURCE_CLUSTER_SENSITIVITY.json","H3_SOURCE_CLUSTER_SENSITIVITY.json",
    "H3P_SOURCE_CLUSTER_SENSITIVITY.json","H4_SOURCE_CLUSTER_SENSITIVITY.json",
    "CLAIM_CONCORDANCE_TABLE.csv","SENSITIVITY_SUMMARY.md",
]
missing=[x for x in required_scientific if not (OUT_ROOT/x).is_file()]
if missing:hard_stop("Stage09 scientific artifact contract incomplete",{"missing":missing})

manifest={
    "schema_version":1,"stage_id":STAGE_ID,"stage_name":STAGE_NAME,
    "protocol_version":PROTOCOL_VERSION,"sensitivity_version":SENSITIVITY_VERSION,
    "cluster_rule_version":CLUSTER_RULE_VERSION,
    "scientific_role":"sensitivity only; never replaces canonical Stages05/06/07/08",
    "upstream_identities":UPSTREAM_IDENTITIES,
    "frozen_banglabait":{"repo_id":EXPECTED_BANGLABAIT_REPO,"revision":EXPECTED_BANGLABAIT_REVISION,
                        "test_sha256":EXPECTED_BANGLABAIT_TEST_SHA256,"frozen_zip_sha256_provenance":EXPECTED_FROZEN_ZIP_SHA256},
    "scientific_contract":{
        "bootstrap_replicates":BOOTSTRAP_REPS,"bootstrap_seed":BOOTSTRAP_SEED,"seeds":SEEDS,"models":MODEL_KEYS,
        "h3_cells":32,"h3_reduced_dimension_test":False,"subset_bh":False,"bridge_analysis":False,
        "h4_budgets":BUDGETS,"h4_primary_budget":PRIMARY_BUDGET,
        "model_training":False,"calibration_fitting":False,"prediction_generation":False,
        "target_training":False,"target_recalibration":False,"new_hypothesis":False,
        "rejection_resampling":False,"survivor_filtering":False,
        "canonical_nonestimability_upgrade_allowed":False,
    },
    "artifacts":{x:sha256_file(OUT_ROOT/x) for x in required_scientific},
}
manifest["manifest_payload_sha256"]=hashlib.sha256(cjb(manifest)).hexdigest()
write_json(OUT_ROOT/"REPRODUCIBILITY_MANIFEST.json",manifest)

entries=[]
for p in sorted(OUT_ROOT.iterdir(),key=lambda x:x.name):
    if p.is_file() and p.name not in {"STAGE09_SHA256SUMS.txt","STAGE_STATUS.json"}:
        entries.append((p.name,sha256_file(p)))
write_text(OUT_ROOT/"STAGE09_SHA256SUMS.txt","\n".join(f"{h}  {name}" for name,h in entries)+"\n")

set_status("PASS","Source-site host cluster sensitivity completed without modifying canonical inference",{
    "source_rows":diag["total_source_test_rows"],"source_clusters":diag["total_publisher_clusters"],
    "global_h3_sensitivity_status":h3s["global_banglaclick_outcome_wald"]["status"],
    "concordance_class_counts":classes,"output_root":str(OUT_ROOT),
})
print("STAGE 09 SOURCE-SITE CLUSTER SENSITIVITY — PASS",flush=True)


STAGE 09 STATUS: PASS — Source-site host cluster sensitivity completed without modifying canonical inference
STAGE 09 SOURCE-SITE CLUSTER SENSITIVITY — PASS
